In [ ]:
# Setting up the API key
import os
# del os.environ['NVIDIA_API_KEY']  ## delete key and reset if needed
if os.environ.get("NGC_API_KEY", "").startswith("nvapi-"):
    print("Valid NGC_API_KEY already in environment. Delete to reset")
else:
    candidate_api_key = getpass("NVAPI Key (starts with nvapi-): ")
    assert candidate_api_key.startswith("nvapi-"), (
        f"{candidate_api_key[:5]}... is not a valid key"
    )
    os.environ["NGC_API_KEY"] = candidate_api_key

In [ ]:
# login to nvcr.io

!echo "${NGC_API_KEY}" | docker login nvcr.io -u '$oauthtoken' --password-stdin

In [ ]:
# Deploy the vector database

!docker compose -f vectordb.yaml up -d

In [ ]:
# Start the ingestor-server
import os

# This are used by nv-ingest-ms-runtime container when using cloud models
os.environ["OCR_HTTP_ENDPOINT"] = "https://ai.api.nvidia.com/v1/cv/nvidia/nemoretriever-ocr"
os.environ["OCR_INFER_PROTOCOL"] = "http"
os.environ["YOLOX_HTTP_ENDPOINT"] = (
    "https://ai.api.nvidia.com/v1/cv/nvidia/nemoretriever-page-elements-v2"
)
os.environ["YOLOX_INFER_PROTOCOL"] = "http"
os.environ["YOLOX_GRAPHIC_ELEMENTS_HTTP_ENDPOINT"] = (
    "https://ai.api.nvidia.com/v1/cv/nvidia/nemoretriever-graphic-elements-v1"
)
os.environ["YOLOX_GRAPHIC_ELEMENTS_INFER_PROTOCOL"] = "http"
os.environ["YOLOX_TABLE_STRUCTURE_HTTP_ENDPOINT"] = (
    "https://ai.api.nvidia.com/v1/cv/nvidia/nemoretriever-table-structure-v1"
)
os.environ["YOLOX_TABLE_STRUCTURE_INFER_PROTOCOL"] = "http"

!docker compose -f docker-compose-ingestor-server.yaml up -d

# Ingestion API Usage

This section demonstrates how to interact with the ingestion APIs to upload and index documents for retrieval-augmented generation (RAG) applications. It showcases the different APIs needed to create a collection, upload documents to the created collection using Milvus Vector DB. It also showcases different APIs to manage uploaded documents and existing collections effectively.



- Ensure the ingestor-server container is running before executing the notebook by following the steps in [Get Started](../docs/deploy-docker-self-hosted.md).
- Replace `BASE_URL` with the actual server URL if the API is hosted on another system.
- You can customize the directory path (`../data/multimodal`) with the correct location of your dataset.


#### 1. Install Dependencies and import required modules

In [ ]:
!uv pip install aiohttp
import json
import os

import aiohttp

#### 2. Setup Base Configuration

In [ ]:
IPADDRESS = "localhost"
INGESTOR_SERVER_PORT = "8082"
BASE_URL = f"http://{IPADDRESS}:{INGESTOR_SERVER_PORT}"  # Replace with your server URL


async def print_response(response):
    """Helper to print API response."""
    try:
        response_json = await response.json()
        print(json.dumps(response_json, indent=2))
    except aiohttp.ClientResponseError:
        print(await response.text())

#### 3. Health Check Endpoint

**Purpose:**
This endpoint performs a health check on the server. It returns a 200 status code if the server is operational.

In [ ]:
async def fetch_health_status():
    """Fetch health status asynchronously."""
    url = f"{BASE_URL}/v1/health"
    params = {"check_dependencies": "True"}
    async with aiohttp.ClientSession() as session:
        async with session.get(url, params=params) as response:
            await print_response(response)


# Run the async function
await fetch_health_status()

#### 4. Create collection Endpoint

**Purpose:**
This endpoint is used to create a collection in the vector store. 

In [ ]:
async def create_collection(
    collection_name: list = None,
    embedding_dimension: int = 2048,
    metadata_schema: list = [],
):
    data = {
        "collection_name": collection_name,
        "embedding_dimension": embedding_dimension,
        "metadata_schema": metadata_schema,
    }

    HEADERS = {"Content-Type": "application/json"}

    async with aiohttp.ClientSession() as session:
        try:
            async with session.post(
                f"{BASE_URL}/v1/collection", json=data, headers=HEADERS
            ) as response:
                await print_response(response)
        except aiohttp.ClientError as e:
            return 500, {"error": str(e)}


# [Optional]: Define schema for metadata fields
metadata_schema = [
    {
        "name": "timestamp",
        "type": "datetime",  # Field of time datetime (i.e string in ISO 8601 format)
        "description": "Following field would store the timestamp of when the document was created",
    },
    {
        "name": "meta_field_1",
        "type": "string",
        "description": "Following field would contain the description for the document",
    },
]

# Call create collection method
await create_collection(
    collection_name="multimodal_data",
    metadata_schema=metadata_schema,  # Optional argument, can be commented if metadata is not to be inserted
)

#### 4. Upload Document Endpoint

**Purpose:**
This endpoint uploads new documents to the vector store. 
1. You can specify the collection name where the documents should be stored. 
2. The collection to which the documents are being uploaded must exist in the vector database.
3. The documents which are uploaded must not exist in the collection. If the documents already exists, to reingest existing files in the provided collection, replace `session.post(...)` with `session.patch(...)`
4. To speed up the ingestion process, the multiple files can be passed in a single request as showcased below.

In [ ]:
# Filepaths
FILEPATHS = [
    "../data/multimodal/embedded_table.pdf",
    "../data/multimodal/functional_validation.pdf",
    "../data/multimodal/woods_frost.pdf",
    "../data/multimodal/multimodal_test.pdf",
    "../data/multimodal/table_test.pdf",
    "../data/multimodal/woods_frost.docx",
]

# [Optional]: Add filename specific custom metadata
# Note: timestamp metadata field must be in ISO 8601 format so following operands are supported: "==", "<=",
CUSTOM_METADATA = [
    {
        "filename": "multimodal_test.pdf",
        "metadata": {
            "timestamp": "2000-05-15T10:23:00",
            "meta_field_1": "multimodal document",
        },
    },
    {
        "filename": "functional_validation.pdf",
        "metadata": {
            "timestamp": "2001-05-15T10:23:00",
            "meta_field_1": "functional validation document",
        },
    },
    {
        "filename": "woods_frost.pdf",
        "metadata": {
            "timestamp": "2002-05-15T10:23:00",
            "meta_field_1": "multimodal document",
        },
    },
]

In [ ]:
async def upload_documents(collection_name: str = ""):
    data = {
        "collection_name": collection_name,
        "blocking": False,  # If True, upload is blocking; else async. Status API not needed when blocking
        "split_options": {"chunk_size": 512, "chunk_overlap": 150},
        "custom_metadata": CUSTOM_METADATA,
        "generate_summary": False,  # Set to True to optionally generate summaries for all documents after ingestion
    }

    form_data = aiohttp.FormData()
    for file_path in FILEPATHS:
        form_data.add_field(
            "documents",
            open(file_path, "rb"),
            filename=os.path.basename(file_path),
            content_type="application/pdf",
        )

    form_data.add_field("data", json.dumps(data), content_type="application/json")

    async with aiohttp.ClientSession() as session:
        try:
            async with session.post(
                f"{BASE_URL}/v1/documents", data=form_data
            ) as response:  # Replace with session.patch for reingesting
                await print_response(response)
                # Return the response JSON for task_id extraction
                response_json = await response.json()
                return response_json
        except aiohttp.ClientError as e:
            print(f"Error: {e}")
            return None

# Store the response and extract task_id
upload_response = await upload_documents(collection_name="multimodal_data")
task_id = upload_response.get("task_id") if upload_response else None
print(f"Extracted task_id: {task_id}")

#### 5. Get Task Status Endpoint:

**Purpose:**

This endpoint is used to get task status of upload documents task. When task is `"FINISHED"`, this endpoint can be used to get status report of the upload task. Uncomment the snippet in the cell below to see document-wise updates.

**Optional:** 

To see granular status updates, either upload more documents or modify the deployment configuration to change file per batch and concurrency to 1:
```bash
export NV_INGEST_FILES_PER_BATCH=1
export NV_INGEST_CONCURRENT_BATCHES=1
```


In [ ]:
import asyncio

async def get_task_status(task_id: str):
    params = {
        "task_id": task_id,
    }

    HEADERS = {"Content-Type": "application/json"}

    async with aiohttp.ClientSession() as session:
        try:
            async with session.get(
                f"{BASE_URL}/v1/status", params=params, headers=HEADERS
            ) as response:
                return await response.json()
        except aiohttp.ClientError as e:
            return 500, {"error": str(e)}

# Use the extracted task_id from the upload_documents response to Poll the status API endpoint
if task_id:
    while True:
        status_response = await get_task_status(task_id=task_id)
        
        state = status_response.get('state', 'N/A')
        result = status_response.get('result', {})
        documents_completed = result.get('documents_completed', 'N/A')
        total_documents = result.get('total_documents', 'N/A')

        print(
            f"-"*100,
            f"\nPolling Task Status - State: {state}, "
            f"Documents Completed: {documents_completed}, "
            f"Total Documents: {total_documents}"
        )
        
        # Uncomment the block below to see document-wise status
        # document_wise_status = status_response.get('document_wise_status', {})
        # if document_wise_status:
        #     print("  Document-wise Status:")
        #     for doc_name, doc_status in document_wise_status.items():
        #         print(f"    • {doc_name}: {doc_status}")
        # print()

        if state == 'FINISHED':
            print("-"*100)
            print(f"Task {task_id} completed successfully")
            print("\nFinal Response:")
            print(json.dumps(status_response, indent=2))
            break
        else:
            await asyncio.sleep(5) # Sleep for 5 seconds before polling again
else:
    print("No task_id available. Please run the upload_documents cell first.")

#### 6. Get Documents Endpoint

**Purpose:**
This endpoint retrieves a list of documents ingested into the vector store for a specified collection.

In [ ]:
async def fetch_documents(collection_name: str = ""):
    url = f"{BASE_URL}/v1/documents"
    params = {"collection_name": collection_name}
    async with aiohttp.ClientSession() as session:
        try:
            async with session.get(url, params=params) as response:
                await print_response(response)
        except aiohttp.ClientError as e:
            print(f"Error: {e}")


await fetch_documents(collection_name="multimodal_data")